# ATLAS evaluation on a free Colab GPU

Runs the attack suites, firewall comparison, repeated-attack test and arena check with a local model. No API key is needed, and all tools and secrets are fake.

1. Choose **Runtime > Change runtime type > T4 GPU**, then Save.
2. Choose **Runtime > Run all**. If Colab warns the notebook isn't from Google, choose **Run anyway**.
3. Keep this tab open. The last cell downloads `atlas-reports.zip`. You can rerun that cell at any time to get results saved so far.

In [ ]:
# Stops here if no GPU is attached, so nothing runs slowly by accident.
import shutil, subprocess
assert shutil.which('nvidia-smi') and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0, 'No GPU. Choose Runtime > Change runtime type > T4 GPU, then Run all again.'
!nvidia-smi -L

In [ ]:
%cd /content
!rm -rf Atlas && git clone --quiet https://github.com/Itzdhruvpatel/Atlas.git
%cd /content/Atlas
!pip install --quiet -e '.[dev]'
!python -m pytest -q 2>&1 | tail -1

In [ ]:
# Installs Ollama, starts it and downloads the model (about 5 GB).
import time, urllib.request
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
subprocess.Popen('OLLAMA_KEEP_ALIVE=-1 nohup ollama serve > /content/ollama.log 2>&1 &', shell=True)
for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/version', timeout=2)
        break
    except OSError:
        time.sleep(2)
else:
    raise RuntimeError('Ollama did not start. Read /content/ollama.log')
MODEL = 'lfm2.5:8b'
BASE = 'http://localhost:11434/v1'
!ollama pull {MODEL} 2>&1 | tail -1
!ollama run {MODEL} 'Say ready' 2>&1 | tail -1
!ollama ps

In [ ]:
# No firewall, several runs each, so rates are less noisy.
!time python -m eval.baseline --suite strong --base-url {BASE} --model {MODEL} --runs 10 --max-llm-calls 600 --timeout 300
!time python -m eval.baseline --base-url {BASE} --model {MODEL} --runs 3 --max-llm-calls 500 --timeout 300

In [ ]:
# Firewall layers on and off. Saves after each setup.
!time python -m eval.ablation --suite strong --base-url {BASE} --model {MODEL} --runs 5 --max-llm-calls 4000 --timeout 300

In [ ]:
# Paper test: the same attack repeated in 1, 2 and 3 documents.
!time python -m eval.h1_spike --base-url {BASE} --model {MODEL} --trials 30 --max-llm-calls 3000 --timeout 300

In [ ]:
# Real rounds through the real arena API, one set per level.
!time python -m eval.arena_check --base-url {BASE} --model {MODEL} --trials 5

In [ ]:
# Downloads every saved report. Rerun this any time to get what has finished so far.
!cd /content/Atlas/eval/reports && zip -q -r /content/atlas-reports.zip . && ls -la
from google.colab import files
files.download('/content/atlas-reports.zip')